# Two-Stage Cascaded Indian Rupee Auto-Labeler
### Model: `Vishva007/gemma-4-12B-it-W4A16-AutoRound-AWQ`
Optimized for **Google Colab GPU L4 (24GB VRAM)**

#### Architecture Overview:
- **Stage 1 (Text-Only Class-Aware Prompt)**: Blazing fast single-image inference (~2.5s/img) with tight, pixel-perfect bounding boxes for 96%+ of banknotes.
- **Stage 2 (Visual Reference Fallback)**: Automatically catches any missed or challenging/folded banknotes using Target-First front & back visual references.
- **Output**: Standard Pascal VOC XML annotations (`labels/<class>/<image>.xml`), colored visual previews (`previews/<class>/<image>.jpg`), and detailed CSV audit report.

## 1. Verify GPU Runtime (L4 GPU Recommended)

In [ ]:
!nvidia-smi

## 2. Install Required Dependencies

In [ ]:
!pip install -q transformers>=4.49.0 accelerate>=1.0.0 gptqmodel>=1.8.0 autoawq>=0.2.7 opencv-python-headless pillow tqdm huggingface_hub

## 3. Prepare Dataset (Crucial for L4 Speed!)
> ⚠️ **IMPORTANT: Do NOT run inference directly on mounted Google Drive folders!**
> Accessing 5,400+ images via Google Drive FUSE adds 1-2 seconds per image file read.
> Always copy or unzip the dataset directly to local Colab NVMe `/content/` disk.

In [ ]:
# Option A: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/indian_currency_640.zip -d /content/

# Option B: Unzip directly uploaded archive
# !unzip -q /content/indian_currency_640.zip -d /content/

## 4. Run the Auto-Labeler (High Throughput Batched Mode)

- `--batch-size 8` (default) or `16`: Utilizes L4's 24GB VRAM and cuts per-image latency from 6s down to ~0.5s!
- `--mode cascade`: Stage 1 batched text prompt -> Stage 2 visual fallback for missed notes.
- `--images-per-class 10`: Quick test on 10 images per class (set `0` to run all 5,400+ images).
- `--skip-existing`: Safe resume if interrupted.

In [ ]:
!python gemma_autoround.py \
  --dataset-dir /content/indian_currency_640 \
  --ref-dir currency_references \
  --mode cascade \
  --batch-size 8 \
  --images-per-class 10 \
  --skip-existing


## 5. Download Generated Annotations & Previews

In [ ]:
!zip -rq /content/annotations_and_previews.zip labels previews annotation_report.csv
from google.colab import files
files.download('/content/annotations_and_previews.zip')